# C12 Tables & Figures Generation
**Result 6: BCR/TCR Repertoire Analysis**

## Output Plan:
**Main Table 8:** BCR/TCR Donor-Level Key Metrics by Stage × Tissue

**Main Figure 8:** BCR/TCR IT-Oriented Key Findings (4 panels)
- A: Blood TCR clonality (★IT-specific)
- B: Liver B_c04-COCH proportion (★Chronic)
- C: Blood IgD% (★Chronic)
- D: Liver SDC1 proportion + IT→IA collapse (★Transition)

**Supp Table S6:** Complete Donor-Level BCR/TCR Metrics

**Supp Table S7:** IT→IA Transition BCR/TCR Results (52 tests)

**Supp Figure S8:** Additional BCR/TCR Panels
- A: Blood BCR top clone size
- B: Liver FCRL5 proportion
- C: Liver plasmaB_c03-MKI67 proportion
- D: BCR Isotype distribution (stacked bar)

In [ ]:
# Cell 1: Setup
from google.colab import drive
drive.mount('/content/drive')

import scanpy as sc
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings; warnings.filterwarnings('ignore')
import os

DATA_PATH = '/content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad'
SAVE_DIR = '/content/drive/MyDrive/ITLAS/results/version18-analysis-v2/BCR_TCR'
FIG_DIR = f'{SAVE_DIR}/figures'
os.makedirs(FIG_DIR, exist_ok=True)

adata = sc.read_h5ad(DATA_PATH, backed='r')
obs = adata.obs.copy()
obs['donor'] = obs['sample'].astype(str).str.split('_').str[1]

# V18 color scheme
LIVER_COLOR = '#D32F2F'  # red
BLOOD_COLOR = '#1565C0'  # blue
STAGE_COLORS = {'NL':'#4CAF50','IT':'#F44336','IA':'#FF9800','AR':'#2196F3','CR':'#9C27B0'}
STAGE_ORDER = ['NL','IT','IA','AR','CR']

def safe_clonality(cc):
    nu=len(cc); nt=cc.sum()
    if nu<=0 or nt<=0: return 0.0
    if nu==1: return 1.0 if nt>1 else 0.0
    fr=cc.values/nt; fr=fr[fr>0]
    ent=-np.sum(fr*np.log2(fr))
    return 1-(ent/np.log2(nu)) if np.log2(nu)>0 else 0.0

print('Setup complete.')

In [ ]:
# Cell 2: Compute all donor-level metrics needed for tables and figures

# --- BCR ---
bcr_rows = []
for (stage, tissue, donor), grp in obs.groupby(['Stage','tissue','donor'], observed=True):
    n = len(grp)
    bcr = grp[grp['BCR_clone.id'].notna()]
    nb = len(bcr)
    if nb == 0:
        bcr_rows.append({'Stage':stage,'tissue':tissue,'donor':donor,
                         'n_bcr':0,'pct_bcr':0,'n_unique':0,
                         'clonality':np.nan,'pct_singleton':np.nan,
                         'pct_IgM':np.nan,'pct_IgG':np.nan,'pct_IgA':np.nan,'pct_IgD':np.nan,
                         'pct_switched':np.nan,'top_clone':0,'pct_IGHV3_23':np.nan})
        continue
    cc=bcr['BCR_clone.id'].value_counts(); nu=len(cc); ns=(cc==1).sum()
    iso=bcr['BCR_CType'].value_counts(); it=iso.sum()
    vg=bcr['BCR_v_gene'].value_counts()
    bcr_rows.append({'Stage':stage,'tissue':tissue,'donor':donor,
                     'n_bcr':nb,'pct_bcr':nb/n*100,'n_unique':nu,
                     'clonality':safe_clonality(cc),'pct_singleton':ns/nu*100,
                     'pct_IgM':iso.get('IGHM',0)/it*100,'pct_IgG':iso.get('IGHG',0)/it*100,
                     'pct_IgA':iso.get('IGHA',0)/it*100,'pct_IgD':iso.get('IGHD',0)/it*100,
                     'pct_switched':(iso.get('IGHG',0)+iso.get('IGHA',0))/it*100,
                     'top_clone':cc.max(),'pct_IGHV3_23':vg.get('IGHV3-23',0)/nb*100})
bcr_df = pd.DataFrame(bcr_rows)

# --- TCR ---
tcr_rows = []
for (stage, tissue, donor), grp in obs.groupby(['Stage','tissue','donor'], observed=True):
    n = len(grp)
    tcr = grp[grp['TCR_clone.id'].notna()]
    nt = len(tcr)
    if nt == 0:
        tcr_rows.append({'Stage':stage,'tissue':tissue,'donor':donor,
                         'n_tcr':0,'pct_tcr':0,'n_unique':0,
                         'clonality':np.nan,'pct_singleton':np.nan,'top_clone':0})
        continue
    cc=tcr['TCR_clone.id'].value_counts(); nu=len(cc); ns=(cc==1).sum()
    tcr_rows.append({'Stage':stage,'tissue':tissue,'donor':donor,
                     'n_tcr':nt,'pct_tcr':nt/n*100,'n_unique':nu,
                     'clonality':safe_clonality(cc),'pct_singleton':ns/nu*100,
                     'top_clone':cc.max()})
tcr_df = pd.DataFrame(tcr_rows)

# --- B/PlasmaB subcluster proportions ---
bp = obs[obs['major_lineage'].isin(['B','PlasmaB'])].copy()
sc_list = sorted(bp['gut2021_subcluster_v2'].unique())
sc_rows = []
for (stage, tissue, donor), grp in bp.groupby(['Stage','tissue','donor'], observed=True):
    total = len(grp)
    if total < 5: continue
    counts = grp['gut2021_subcluster_v2'].value_counts()
    for sc in sc_list:
        sc_rows.append({'Stage':stage,'tissue':tissue,'donor':donor,
                        'subcluster':sc,'proportion':counts.get(sc,0)/total*100})
sc_df = pd.DataFrame(sc_rows)

print(f'BCR donor records: {len(bcr_df)}')
print(f'TCR donor records: {len(tcr_df)}')
print(f'Subcluster proportion records: {len(sc_df)}')

In [ ]:
# Cell 3: MAIN TABLE 8 — BCR/TCR Key Metrics Summary
print('='*70)
print('TABLE 8: BCR/TCR Donor-Level Repertoire Metrics')
print('='*70)

# BCR table
print('\nPanel A: BCR Repertoire')
print(f'{"":<8}{"":<8}{"n":>5}{"BCR cells":>12}{"Clonality":>12}{"Singleton%":>12}'
      f'{"IgM%":>8}{"IgG%":>8}{"IgA%":>8}{"IgD%":>8}{"Switched%":>10}{"TopClone":>10}')

table8_rows = []
for tissue_val in ['Liver','Blood']:
    for stage in STAGE_ORDER:
        s = bcr_df[(bcr_df['Stage']==stage)&(bcr_df['tissue']==tissue_val)&(bcr_df['n_bcr']>0)]
        if len(s)==0:
            print(f'{tissue_val:<8}{stage:<8}{"—":>5}')
            table8_rows.append({'Tissue':tissue_val,'Stage':stage,'n_donors':0})
            continue
        row = {'Tissue':tissue_val,'Stage':stage,'n_donors':len(s),
               'BCR_cells':f'{s.n_bcr.mean():.0f}±{s.n_bcr.std():.0f}',
               'Clonality':f'{s.clonality.mean():.3f}±{s.clonality.std():.3f}',
               'Singleton':f'{s.pct_singleton.mean():.1f}',
               'IgM':f'{s.pct_IgM.mean():.1f}','IgG':f'{s.pct_IgG.mean():.1f}',
               'IgA':f'{s.pct_IgA.mean():.1f}','IgD':f'{s.pct_IgD.mean():.1f}',
               'Switched':f'{s.pct_switched.mean():.1f}',
               'TopClone':f'{s.top_clone.mean():.1f}'}
        table8_rows.append(row)
        print(f'{tissue_val:<8}{stage:<8}{len(s):>5}{row["BCR_cells"]:>12}{row["Clonality"]:>12}'
              f'{row["Singleton"]:>12}{row["IgM"]:>8}{row["IgG"]:>8}{row["IgA"]:>8}'
              f'{row["IgD"]:>8}{row["Switched"]:>10}{row["TopClone"]:>10}')

# TCR table
print('\nPanel B: TCR Repertoire')
print(f'{"":<8}{"":<8}{"n":>5}{"TCR cells":>12}{"Clonality":>12}{"Singleton%":>12}{"TopClone":>10}')

for tissue_val in ['Liver','Blood']:
    for stage in STAGE_ORDER:
        s = tcr_df[(tcr_df['Stage']==stage)&(tcr_df['tissue']==tissue_val)&(tcr_df['n_tcr']>0)]
        if len(s)==0:
            print(f'{tissue_val:<8}{stage:<8}{"—":>5}')
            continue
        print(f'{tissue_val:<8}{stage:<8}{len(s):>5}'
              f'{s.n_tcr.mean():.0f}±{s.n_tcr.std():.0f}:>12'
              f'{s.clonality.mean():.3f}±{s.clonality.std():.3f}:>12'
              f'{s.pct_singleton.mean():.1f}:>12'
              f'{s.top_clone.mean():.0f}:>10')

In [ ]:
# Cell 3b: TABLE 8 — Clean Excel export
import openpyxl

# Panel A: BCR
bcr_table_rows = []
for tissue_val in ['Liver','Blood']:
    for stage in STAGE_ORDER:
        s = bcr_df[(bcr_df['Stage']==stage)&(bcr_df['tissue']==tissue_val)&(bcr_df['n_bcr']>0)]
        if len(s)==0:
            bcr_table_rows.append({'Tissue':tissue_val,'Stage':stage,'n':0})
            continue
        bcr_table_rows.append({
            'Tissue':tissue_val,'Stage':stage,'n':len(s),
            'BCR_cells_mean':round(s.n_bcr.mean(),0),
            'BCR_cells_sd':round(s.n_bcr.std(),0),
            'Clonality_mean':round(s.clonality.mean(),4),
            'Clonality_sd':round(s.clonality.std(),4),
            'Singleton_pct':round(s.pct_singleton.mean(),1),
            'IgM_pct':round(s.pct_IgM.mean(),1),
            'IgG_pct':round(s.pct_IgG.mean(),1),
            'IgA_pct':round(s.pct_IgA.mean(),1),
            'IgD_pct':round(s.pct_IgD.mean(),1),
            'ClassSwitched_pct':round(s.pct_switched.mean(),1),
            'TopCloneSize':round(s.top_clone.mean(),1),
            'IGHV3_23_pct':round(s.pct_IGHV3_23.mean(),1),
        })

# Panel B: TCR
tcr_table_rows = []
for tissue_val in ['Liver','Blood']:
    for stage in STAGE_ORDER:
        s = tcr_df[(tcr_df['Stage']==stage)&(tcr_df['tissue']==tissue_val)&(tcr_df['n_tcr']>0)]
        if len(s)==0:
            tcr_table_rows.append({'Tissue':tissue_val,'Stage':stage,'n':0})
            continue
        tcr_table_rows.append({
            'Tissue':tissue_val,'Stage':stage,'n':len(s),
            'TCR_cells_mean':round(s.n_tcr.mean(),0),
            'TCR_cells_sd':round(s.n_tcr.std(),0),
            'Clonality_mean':round(s.clonality.mean(),4),
            'Clonality_sd':round(s.clonality.std(),4),
            'Singleton_pct':round(s.pct_singleton.mean(),1),
            'TopCloneSize':round(s.top_clone.mean(),0),
        })

# Save as Excel
with pd.ExcelWriter(f'{SAVE_DIR}/Table8_BCR_TCR_Summary.xlsx') as writer:
    pd.DataFrame(bcr_table_rows).to_excel(writer, sheet_name='Panel_A_BCR', index=False)
    pd.DataFrame(tcr_table_rows).to_excel(writer, sheet_name='Panel_B_TCR', index=False)
print('Saved: Table8_BCR_TCR_Summary.xlsx')

In [ ]:
# Cell 4: MAIN FIGURE 8 — 4-panel donor dot plots
# V18 rules: Liver=red circle, Blood=blue triangle, p-values shown, dot plots only

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Figure 8. BCR/TCR Repertoire Analysis: IT-Oriented Key Findings',
             fontsize=14, fontweight='bold', y=0.98)

def donor_dot_plot(ax, data_df, metric, tissue_val, title, ylabel,
                   p_annotations=None, stages=['NL','IT','IA','AR','CR']):
    """Donor-level dot plot with group means."""
    present_stages = [s for s in stages if s in data_df['Stage'].values]
    is_liver = tissue_val == 'Liver'
    color = LIVER_COLOR if is_liver else BLOOD_COLOR
    marker = 'o' if is_liver else '^'
    
    for i, stage in enumerate(present_stages):
        vals = data_df[(data_df['Stage']==stage)&(data_df['tissue']==tissue_val)]
        if metric in vals.columns:
            y = vals[metric].dropna()
        else:
            continue
        if len(y) == 0:
            continue
        # Jitter
        x = np.random.normal(i, 0.08, size=len(y))
        ax.scatter(x, y, c=color, marker=marker, s=50, alpha=0.7, edgecolors='white', linewidth=0.5)
        # Mean bar
        ax.plot([i-0.2, i+0.2], [y.mean(), y.mean()], color=color, linewidth=2.5)
    
    ax.set_xticks(range(len(present_stages)))
    ax.set_xticklabels(present_stages, fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=11)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # P-value annotations
    if p_annotations:
        ymax = ax.get_ylim()[1]
        for pa in p_annotations:
            x1, x2, p_val, p_str = pa
            y_bar = ymax * 0.88
            ax.plot([x1, x1, x2, x2], [y_bar*0.95, y_bar, y_bar, y_bar*0.95],
                    color='black', linewidth=1)
            ax.text((x1+x2)/2, y_bar*1.01, p_str, ha='center', va='bottom', fontsize=9,
                    color='#D32F2F' if p_val<0.05 else '#666666')

# --- Panel A: Blood TCR Clonality (IT-specific ★) ---
ax = axes[0, 0]
donor_dot_plot(ax, tcr_df[tcr_df['n_tcr']>0], 'clonality', 'Blood',
              'A. Blood TCR Clonality — IT-Specific', 'Clonality',
              p_annotations=[(0, 1, 0.032, '★p=0.032')])

# --- Panel B: Liver B_c04-COCH (Chronic ★) ---
ax = axes[0, 1]
coch = sc_df[sc_df['subcluster']=='B_c04-COCH']
donor_dot_plot(ax, coch, 'proportion', 'Liver',
              'B. Liver B_c04-COCH Proportion', '% of B+PlasmaB',
              p_annotations=[(0, 1, 0.014, '★p=0.014')])

# --- Panel C: Blood IgD% (Chronic ★) ---
ax = axes[1, 0]
donor_dot_plot(ax, bcr_df[bcr_df['n_bcr']>0], 'pct_IgD', 'Blood',
              'C. Blood BCR IgD% — Naive B Depletion', 'IgD (%)',
              p_annotations=[(0, 1, 0.016, '★p=0.016')])

# --- Panel D: Liver SDC1 (IT→IA Transition ★) ---
ax = axes[1, 1]
sdc1 = sc_df[sc_df['subcluster']=='plasmaB_c01-SDC1']
donor_dot_plot(ax, sdc1, 'proportion', 'Liver',
              'D. Liver SDC1+ Plasma Cells — IT→IA Collapse', '% of B+PlasmaB',
              p_annotations=[(1, 2, 0.008, '★p=0.008')])

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(f'{FIG_DIR}/Figure8_BCR_TCR_key_findings.png', dpi=300, bbox_inches='tight')
plt.savefig(f'{FIG_DIR}/Figure8_BCR_TCR_key_findings.pdf', bbox_inches='tight')
plt.show()
print('Saved: Figure8_BCR_TCR_key_findings.png/pdf')

In [ ]:
# Cell 5: SUPPLEMENTARY FIGURE S8 — Additional panels

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Supplementary Figure S8. Additional BCR/TCR Findings',
             fontsize=13, fontweight='bold', y=0.98)

# --- Panel A: Blood BCR top clone size (★ Chronic) ---
ax = axes[0, 0]
donor_dot_plot(ax, bcr_df[bcr_df['n_bcr']>0], 'top_clone', 'Blood',
              'A. Blood BCR Top Clone Size', 'Cells in largest clone',
              p_annotations=[(0, 1, 0.006, '★p=0.006')])

# --- Panel B: Liver FCRL5 proportion († trend) ---
ax = axes[0, 1]
fcrl5 = sc_df[sc_df['subcluster']=='B_c07-FCRL5']
donor_dot_plot(ax, fcrl5, 'proportion', 'Liver',
              'B. Liver B_c07-FCRL5 (Exhausted B)', '% of B+PlasmaB',
              p_annotations=[(0, 1, 0.052, '†p=0.052')])

# --- Panel C: Liver MKI67 proportion († trend) ---
ax = axes[1, 0]
mki67 = sc_df[sc_df['subcluster']=='plasmaB_c03-MKI67']
donor_dot_plot(ax, mki67, 'proportion', 'Liver',
              'C. Liver plasmaB_c03-MKI67 (Proliferating)', '% of B+PlasmaB',
              p_annotations=[(0, 1, 0.082, '†p=0.082')])

# --- Panel D: BCR Isotype Distribution (stacked bar) ---
ax = axes[1, 1]
iso_data = []
for tissue_val in ['Liver','Blood']:
    for stage in ['NL','IT','IA','AR']:
        s = bcr_df[(bcr_df['Stage']==stage)&(bcr_df['tissue']==tissue_val)&(bcr_df['n_bcr']>0)]
        if len(s)==0: continue
        iso_data.append({
            'label': f'{tissue_val}\n{stage}',
            'IgM': s.pct_IgM.mean(), 'IgG': s.pct_IgG.mean(),
            'IgA': s.pct_IgA.mean(), 'IgD': s.pct_IgD.mean()
        })
iso_plot = pd.DataFrame(iso_data)
x = range(len(iso_plot))
colors_iso = {'IgM':'#42A5F5','IgG':'#EF5350','IgA':'#66BB6A','IgD':'#FFA726'}
bottom = np.zeros(len(iso_plot))
for isotype, color in colors_iso.items():
    vals = iso_plot[isotype].values
    ax.bar(x, vals, bottom=bottom, color=color, label=isotype, width=0.7, edgecolor='white')
    bottom += vals
ax.set_xticks(x)
ax.set_xticklabels(iso_plot['label'], fontsize=8, rotation=0)
ax.set_ylabel('Isotype Distribution (%)', fontsize=11)
ax.set_title('D. BCR Isotype by Stage × Tissue', fontsize=12, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
# Add tissue separator
ax.axvline(3.5, color='grey', linestyle='--', alpha=0.5)
ax.text(1.5, 105, 'Liver', ha='center', fontsize=10, color=LIVER_COLOR, fontweight='bold')
ax.text(5.5, 105, 'Blood', ha='center', fontsize=10, color=BLOOD_COLOR, fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(f'{FIG_DIR}/FigS8_BCR_TCR_additional.png', dpi=300, bbox_inches='tight')
plt.savefig(f'{FIG_DIR}/FigS8_BCR_TCR_additional.pdf', bbox_inches='tight')
plt.show()
print('Saved: FigS8_BCR_TCR_additional.png/pdf')

In [ ]:
# Cell 6: SUPPLEMENTARY TABLES S6 & S7

# S6: Complete donor-level BCR/TCR metrics
with pd.ExcelWriter(f'{SAVE_DIR}/SuppTable_S6_donor_level_BCR_TCR.xlsx') as writer:
    bcr_df.to_excel(writer, sheet_name='BCR_donor_level', index=False)
    tcr_df.to_excel(writer, sheet_name='TCR_donor_level', index=False)
    sc_df.to_excel(writer, sheet_name='B_PlasmaB_subcluster_prop', index=False)
print('Saved: SuppTable_S6_donor_level_BCR_TCR.xlsx (3 sheets)')

# S7: IT→IA transition results
trans_path = f'{SAVE_DIR}/C12b_IT_IA_transition_ALL.csv'
if os.path.exists(trans_path):
    trans_df = pd.read_csv(trans_path)
    trans_df.to_excel(f'{SAVE_DIR}/SuppTable_S7_IT_IA_transition_BCR_TCR.xlsx', index=False)
    print('Saved: SuppTable_S7_IT_IA_transition_BCR_TCR.xlsx')
else:
    print(f'⚠️ {trans_path} not found — run C12b notebook first')

# Also save C12 master results
master_path = f'{SAVE_DIR}/MASTER_all_MW_results.csv'
if os.path.exists(master_path):
    master_df = pd.read_csv(master_path)
    master_sig = master_df[master_df['p_value']<0.10] if 'p_value' in master_df.columns else pd.DataFrame()
    with pd.ExcelWriter(f'{SAVE_DIR}/SuppTable_S6b_NL_IT_all_results.xlsx') as writer:
        master_df.to_excel(writer, sheet_name='All_56_tests', index=False)
        if len(master_sig)>0:
            master_sig.to_excel(writer, sheet_name='Significant_p010', index=False)
    print('Saved: SuppTable_S6b_NL_IT_all_results.xlsx')

In [ ]:
# Cell 7: LEGENDS AND FOOTNOTES
legends = """
============================================================
FIGURE LEGENDS
============================================================

Figure 8. BCR/TCR Repertoire Analysis Reveals Clonal Arrest with Active B Cell Engagement in IT.
(A) TCR clonality (1 − normalized Shannon entropy) in blood across disease groups.
IT showed significantly reduced clonal diversity compared with NL
(NL 0.585 → IT 0.351, ★p=0.032, 23/25 donor-pair consistency),
representing the only IT-specific BCR/TCR finding (NL→IA p=0.064, NS).
Each dot represents one donor; horizontal bars indicate group means.
Blue triangles = blood. Mann-Whitney U test; ★p<0.05.
(B) B_c04-COCH proportion within B+PlasmaB cells in liver.
COCH+ B cells showed chronic-persistent expansion from IT
(NL 1.3% → IT 7.1%, ★p=0.014, 29/30; NL→IA ★p=0.022),
indicating active B cell engagement beginning in IT and persisting through IA.
Red circles = liver.
(C) IgD isotype percentage among BCR+ cells in blood.
IgD+ (naive/unswitched) B cells showed chronic-persistent depletion
(NL 17.7% → IT 2.2%, ★p=0.016, 24/25; NL→IA ★p=0.016),
indicating accelerated transition from naive to activated B cell compartment.
(D) plasmaB_c01-SDC1 proportion within B+PlasmaB cells in liver.
Mature plasma cells (SDC1/CD138+) were maintained in IT (NL 32.5% vs IT 26.0%, NS)
but collapsed at the IT→IA transition (IT 26.0% → IA 5.0%, ★p=0.008, 25/25 consistency),
representing a transition event rather than an IT-specific feature.
Panels: 4 (A–D)


Supplementary Figure S8. Additional BCR/TCR Repertoire Findings.
(A) Maximum BCR clone size in blood across disease groups.
IT showed significantly increased top clone size compared with NL
(NL 1.0 → IT 3.0 cells, ★p=0.006, 25/25), a chronic-persistent pattern
(NL→IA ★p=0.010), indicating minimal but statistically significant
clonal expansion in blood BCR beginning at IT.
(B) B_c07-FCRL5 (exhausted/atypical memory B cell) proportion in liver.
FCRL5+ B cells showed a trend toward IT enrichment
(NL 13.0% → IT 28.2%, †p=0.052, 26/30), suggesting
accumulation of functionally impaired B cells in the hepatic microenvironment.
(C) plasmaB_c03-MKI67 (proliferating plasma cell) proportion in liver.
MKI67+ plasma cells showed a trend toward IT depletion
(NL 8.2% → IT 1.4%, †p=0.082, 25/30), consistent with
suppressed plasma cell proliferation during the IT phase.
(D) BCR isotype distribution (IgM, IgG, IgA, IgD) by stage and tissue.
Liver showed higher class-switched (IgG+IgA) proportions than blood across
all stages. In blood, IgD depletion at IT (★p=0.016) was the most prominent
isotype shift. Vertical dashed line separates liver (left) from blood (right).
Blue triangles = blood; red circles = liver. Mann-Whitney U test; ★p<0.05, †p<0.10.
Panels: 4 (A–D)


============================================================
TABLE FOOTNOTES
============================================================

Table 8. BCR/TCR Donor-Level Repertoire Metrics Across the HBV Disease Spectrum.
Panel A: BCR repertoire metrics. Panel B: TCR repertoire metrics.
Values represent donor-level means ± SD. Clonality = 1 − (Shannon entropy / log2(n_unique_clones));
range 0 (all singletons) to 1 (single dominant clone).
Singleton% = percentage of clones represented by a single cell.
Class-switched% = IgG% + IgA%. TopCloneSize = number of cells in the largest clone.
BCR data represent heavy chain (IGH) only; TCR data represent alpha chain (TRA) only.
CR group had no BCR/TCR V(D)J library data available.
AR data (n=1 for liver; n=1 for blood in BCR; n=1 each in TCR) are shown
for reference but excluded from statistical testing due to insufficient sample size.
★p<0.05; †p<0.10; Mann-Whitney U test, two-sided.
Abbreviations: NL, normal liver; IT, immune tolerant; IA, immune active;
AR, acute resolved; CR, chronic resolved.


Supplementary Table S6. Complete Donor-Level BCR/TCR Metrics.
Individual donor values for all BCR (Sheet 1) and TCR (Sheet 2) repertoire metrics,
and B/PlasmaB subcluster proportions (Sheet 3), separated by tissue (liver, blood)
and disease group. These data underlie Table 8 and Figures 8, S8.
Clonality calculated using safe Shannon entropy (log2(0) protected).
n_unique_clones = number of distinct clonotypes per donor per tissue.


Supplementary Table S7. IT→IA Transition BCR/TCR Analysis.
Mann-Whitney U test results for 52 comparisons across BCR repertoire (22 tests),
TCR repertoire (10 tests), and B/PlasmaB subcluster proportions (20 tests),
each tested in liver and blood separately. For each metric, three comparisons
are shown: NL→IT, IT→IA, and NL→IA. Pattern classification:
IT-REVERSAL (NL→IT and IT→IA significant, opposite directions),
IT-AMPLIFIED (same directions), IT-SPECIFIC-RESOLVED (NL→IT sig, NL→IA NS),
IA-EMERGENT (NL→IT NS, IT→IA and NL→IA sig), TRANSITION (IT→IA sig only),
CHRONIC-PERSISTENT (NL→IT and NL→IA sig), NS (none significant).
★p<0.05; †p<0.10; Mann-Whitney U test, two-sided.
"""

# Save legends to file
with open(f'{SAVE_DIR}/C12_Figure_Legends_and_Table_Footnotes.txt', 'w') as f:
    f.write(legends)
print('Saved: C12_Figure_Legends_and_Table_Footnotes.txt')
print(legends)

In [ ]:
# Cell 8: Final inventory
print('='*70)
print('C12 OUTPUT INVENTORY')
print('='*70)
print(f'\nDirectory: {SAVE_DIR}')
print(f'Figures:   {FIG_DIR}')
print()

all_files = []
for root, dirs, files in os.walk(SAVE_DIR):
    for f in sorted(files):
        fpath = os.path.join(root, f)
        sz = os.path.getsize(fpath)
        rel = os.path.relpath(fpath, SAVE_DIR)
        sz_str = f'{sz/1024:.1f}KB' if sz<1024*1024 else f'{sz/1024/1024:.1f}MB'
        all_files.append(rel)
        print(f'  {rel} ({sz_str})')

print(f'\nTotal files: {len(all_files)}')
print('\n✅ All C12 tables, figures, legends, and supplementary materials generated.')